# PyTest: hands-on from the supplied PPT
Based on **2. PyTest.pptx**, following its teaching order. Upload this notebook to [Google Colab](https://colab.research.google.com/) using **File → Upload notebook**, then run cells from top to bottom. CPU / Colab Free is sufficient. Installation requires internet; examples need no external data or credentials.

**Prerequisites:** Python functions, data structures, exceptions, and basic classes. **Suggested duration:** 2–3 hours.

| Slides | Notebook section |
|---|---|
| 3–12 | Setup, discovery, basic tests and failure reports |
| 13 | Passed, failed, skipped, xfail, xpass and errors |
| 14 | Activity 1: Fibonacci tests |
| 15–19 | String, list and dictionary assertion differences |
| 20–22 | Exceptions and Activity 2 |
| 23–27 | Marks, parameters, selection, class/module marks, configuration |
| 28–31 | Fixtures, Activity 3 TV series, conftest.py |
| 32 | No slide text to convert |

**Adaptation notes:** Runnable supporting functions have been added where slides show partial examples. The PPT does not define the Fibonacci function contract, so Activity 1 explicitly assumes `fibo(n)` returns the first `n` values starting at 0. Supplied series values are retained as training data. Optional mocking and TDD extensions at the end support your original topic list; these are additions to the PPT.

**Technical clarifications:** pytest supports unit and integration testing as well as API testing. Default discovery uses `test_*.py` / `*_test.py`, `test_*` functions, and `Test*` classes without custom constructors; configuration can change the patterns. Timeout enforcement needs the `pytest-timeout` plugin. Expected failures below are deliberate and checked, allowing **Run all** to finish.

## How to use this notebook

Every activity explains **what we are doing**, **why it matters**, and **what result to expect**. Read the explanation, run the file-writing cell, then run the test cell. Code comments explain the important lines.

- `%%writefile tests/test_one.py` saves the Python code below it as a file. It does not run the tests.
- `run_tests("tests/test_one.py")` executes that file using pytest.
- `assert` checks whether a condition is true. A false condition fails the test.
- Deliberate failures are learning activities. Their cells explicitly say that a failure is expected.
- Try the exercise before running its solution. Rerun a writing cell after changing its code.
- Run setup again after a Colab runtime restart. Runtime files are temporary.

**Simple test pattern:** Arrange the inputs, Act by calling the function, Assert the expected result. Start with Section 1. The helper functions are provided so you can focus on pytest.

In [ ]:
%pip -q install "pytest>=8,<10" "pytest-timeout>=2,<3"

## 1. Setup and discovery (slides 3–12)
Colab already provides a Python runtime, so a separate virtual environment is unnecessary here. We write test files into a unique practice folder. Each command runs pytest in a fresh subprocess so editing a file does not leave an old imported version in memory.

In [ ]:
from pathlib import Path
import tempfile, textwrap, subprocess, sys, os
LAB = Path(tempfile.mkdtemp(prefix="ppt_pytest_"))
# All relative file paths in the notebook now use this practice folder.
for folder in ("tests", "demos", "practice"):
    (LAB / folder).mkdir()
os.chdir(LAB)
print("Practice folder:", LAB)
def write_file(name, content):
    path = LAB / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip(), encoding="utf-8")
    print("Wrote", name)
# A convenience helper: you do not need to memorize its implementation.
# It runs the equivalent of python -m pytest, prints the report, and checks
# whether this was an expected success (0) or deliberate failure (1).
def run_tests(target=None, expected=0, options=()):
    env = os.environ.copy()
    # Keep unrelated preinstalled plugins from changing our lesson behavior.
    env["PYTEST_DISABLE_PLUGIN_AUTOLOAD"] = "1"
    command = [sys.executable, "-m", "pytest", "-p", "pytest_timeout", *options]
    if target: command.append(target)
    result = subprocess.run(command, cwd=LAB, env=env, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr: print(result.stderr)
    assert result.returncode == expected, f"Expected exit {expected}, got {result.returncode}"
    print("Expected outcome confirmed.")

### Configure our practice project

**What:** Create or update `pytest.ini` in our practice folder.

**Why:** Register custom markers to catch spelling mistakes and select only completed tests by default.

**Expected result:** This file is configuration, not Python. Deliberate failures stay in demos; your optional work stays in practice.

Run this cell to save the file. The following test cell executes it. `%%writefile` is a notebook command: the lines below it become the contents of the named file.

In [ ]:
%%writefile pytest.ini
[pytest]
testpaths = tests
addopts = --strict-markers
markers =
    slow: computation demonstrations
    smoke: quick essential checks
    serial: demonstrations to run sequentially


### Write our first passing test

**What:** Create or update `tests/test_one.py` in our practice folder.

**Why:** A small assertion makes the connection between a requirement and a test result clear.

**Expected result:** The value 75 is below 100, so one test should pass.

Run this cell to save the file. The following test cell executes it. `%%writefile` is a notebook command: the lines below it become the contents of the named file.

In [ ]:
%%writefile tests/test_one.py
marks = 75

# Pytest discovers this test because its name starts with test_.
def test_passing():
    # Compare actual behavior with the expected result; a mismatch fails the test.
    assert marks < 100


**What:** Run the saved test files.

**Why:** Writing a test file alone does not execute its tests. This command asks pytest to collect and run them.

**Expected result:** A successful run. With `--collect-only`, tests are listed without being executed. With a selection option, only matching tests run.

In [ ]:
# Execute pytest in a fresh process and check its exit code.
run_tests('tests/test_one.py', expected=0, options=())

**What:** Run the saved test files.

**Why:** Writing a test file alone does not execute its tests. This command asks pytest to collect and run them.

**Expected result:** A successful run. With `--collect-only`, tests are listed without being executed. With a selection option, only matching tests run.

In [ ]:
# Execute pytest in a fresh process and check its exit code.
run_tests('tests/test_one.py', expected=0, options=('-v',))

The `.` means a passed test. `[100%]` shows completion, not a quality or coverage score. `-v` shows the individual test name. With no path, our configuration selects the `tests` directory. An explicit file overrides that directory selection.

### Observe an intentional test failure

**What:** Create or update `demos/test_two.py` in our practice folder.

**Why:** The traceback identifies the unequal tuple items and the assertion that failed.

**Expected result:** One test fails because the marks appear in a different order. This failure is expected.

Run this cell to save the file. The following test cell executes it. `%%writefile` is a notebook command: the lines below it become the contents of the named file.

In [ ]:
%%writefile demos/test_two.py
marks = (75, 80, 90)

# Pytest discovers this test because its name starts with test_.
def test_failing():
    # Compare actual behavior with the expected result; a mismatch fails the test.
    assert marks == (90, 80, 75)


**What:** Run the saved test files.

**Why:** Writing a test file alone does not execute its tests. This command asks pytest to collect and run them.

**Expected result:** A deliberate failure or setup error. Read the report. The notebook continues because this outcome is expected.

In [ ]:
# Execute pytest in a fresh process and check its exit code.
run_tests('demos/test_two.py', expected=1, options=())

Read the assertion and the first differing tuple index. **Practice:** Explain why these tuples are unequal. If the contract requires ascending marks, the expected tuple must be `(75, 80, 90)`.

A helper without a `test_` prefix is not discovered under the defaults. `--collect-only` lists tests without running them.

### Compare a helper, a test function, and a test class

**What:** Create or update `tests/test_discovery.py` in our practice folder.

**Why:** Naming conventions let pytest find tests automatically.

**Expected result:** The helper is not collected as a test. The function and class method are collected.

Run this cell to save the file. The following test cell executes it. `%%writefile` is a notebook command: the lines below it become the contents of the named file.

In [ ]:
%%writefile tests/test_discovery.py
def helper():
    return 5

# Pytest discovers this test because its name starts with test_.
def test_helper_result():
    # Compare actual behavior with the expected result; a mismatch fails the test.
    assert helper() == 5

# Group related tests; pytest discovers classes starting with Test.
class TestNumbers:
    # Pytest discovers this test because its name starts with test_.
    def test_addition(self):
        # Compare actual behavior with the expected result; a mismatch fails the test.
        assert 2 + 3 == 5


**What:** Run the saved test files.

**Why:** Writing a test file alone does not execute its tests. This command asks pytest to collect and run them.

**Expected result:** A successful run. With `--collect-only`, tests are listed without being executed. With a selection option, only matching tests run.

In [ ]:
# Execute pytest in a fresh process and check its exit code.
run_tests('tests/test_discovery.py', expected=0, options=('--collect-only', '-q'))

## 2. Test outcomes (slide 13)
| Symbol | Meaning |
|---|---|
| `.` | PASSED: assertions succeed |
| `F` | FAILED: a test fails, including an unexpected exception in its body |
| `s` | SKIPPED: a skip condition applies |
| `x` | XFAIL: expected failure occurs |
| `X` | XPASS: a non-strict expected-failure test passes |
| `E` | ERROR: setup/teardown fails; collection can also error |

`xfail(strict=True)` makes an unexpected pass fail the suite. Use skip and xfail with reasons, rather than to conceal defects.

### Demonstrate skip, xfail, xpass, and a setup error

**What:** Create or update `demos/test_outcomes.py` in our practice folder.

**Why:** Recognizing each outcome helps you interpret a test report correctly.

**Expected result:** Two skipped, one xfail, one xpass, and one setup error. This is an intentional demonstration.

Run this cell to save the file. The following test cell executes it. `%%writefile` is a notebook command: the lines below it become the contents of the named file.

In [ ]:
%%writefile demos/test_outcomes.py
# Import pytest so we can use fixtures, marks, and exception checks.
import pytest

# Skip this demonstration and show the reason in the report.
@pytest.mark.skip(reason="Demonstration: feature unavailable")
# Pytest discovers this test because its name starts with test_.
def test_skipped():
    # Compare actual behavior with the expected result; a mismatch fails the test.
    assert False

# Skip this test when the specified condition is true.
@pytest.mark.skipif(True, reason="Demonstration: explicit condition")
# Pytest discovers this test because its name starts with test_.
def test_conditionally_skipped():
    # Compare actual behavior with the expected result; a mismatch fails the test.
    assert False

# Record this as a known expected failure.
@pytest.mark.xfail(reason="Demonstration: known incorrect result")
# Pytest discovers this test because its name starts with test_.
def test_expected_failure():
    # Compare actual behavior with the expected result; a mismatch fails the test.
    assert 2 + 2 == 5

# Record this as a known expected failure.
@pytest.mark.xfail(reason="Demonstration: behavior already fixed", strict=False)
# Pytest discovers this test because its name starts with test_.
def test_unexpected_pass():
    # Compare actual behavior with the expected result; a mismatch fails the test.
    assert 2 + 2 == 4

# Turn this function into reusable test setup; default scope is one test.
@pytest.fixture
def broken_resource():
    # Reject invalid input with a clear exception.
    raise RuntimeError("Deliberate setup failure")

# Pytest discovers this test because its name starts with test_.
def test_setup_error(broken_resource):
    # Compare actual behavior with the expected result; a mismatch fails the test.
    assert broken_resource


**What:** Run the saved test files.

**Why:** Writing a test file alone does not execute its tests. This command asks pytest to collect and run them.

**Expected result:** A deliberate failure or setup error. Read the report. The notebook continues because this outcome is expected.

In [ ]:
# Execute pytest in a fresh process and check its exit code.
run_tests('demos/test_outcomes.py', expected=1, options=('-ra',))

## 3. Activity 1: correct Fibonacci tests (slide 14)
**Assumed contract:** `fibo(n)` returns the first `n` Fibonacci values, starting `[0, 1, 1, 2, ...]`. `n=0` returns `[]`; negative values raise `ValueError`. Assume `n` is an integer.

Predict the correct result for `n=6`. Run the wrong test below, then correct its expected list. The solution follows.

### Create the Fibonacci function used in Activity 1

**What:** Create or update `fibo.py` in our practice folder.

**Why:** Tests need an actual function to call. We define its contract before judging the expected result.

**Expected result:** fibo(6) will return [0, 1, 1, 2, 3, 5].

Run this cell to save the file. The following test cell executes it. `%%writefile` is a notebook command: the lines below it become the contents of the named file.

In [ ]:
%%writefile fibo.py
def fibo(n):
    if n < 0:
        # Reject invalid input with a clear exception.
        raise ValueError("n must be non-negative")
    values = []
    a, b = 0, 1
    for _ in range(n):
        values.append(a)
        a, b = b, a + b
    return values


### Run the incorrect Fibonacci test

**What:** Create or update `demos/test_fibo_wrong.py` in our practice folder.

**Why:** This activity teaches you to check an expected result against the agreed contract.

**Expected result:** The test fails because its expected sequence starts with 1 instead of 0.

Run this cell to save the file. The following test cell executes it. `%%writefile` is a notebook command: the lines below it become the contents of the named file.

In [ ]:
%%writefile demos/test_fibo_wrong.py
# Import the function or class that this test needs.
from fibo import fibo

# Pytest discovers this test because its name starts with test_.
def test_six_values():
    # Compare actual behavior with the expected result; a mismatch fails the test.
    assert fibo(6) == [1, 1, 2, 3, 5, 8]


**What:** Run the saved test files.

**Why:** Writing a test file alone does not execute its tests. This command asks pytest to collect and run them.

**Expected result:** A deliberate failure or setup error. Read the report. The notebook continues because this outcome is expected.

In [ ]:
# Execute pytest in a fresh process and check its exit code.
run_tests('demos/test_fibo_wrong.py', expected=1, options=())

### Activity 1 solution
Starting at 0 gives `[0, 1, 1, 2, 3, 5]`. Add empty and single-element cases to check boundaries.

### Write the corrected Fibonacci tests

**What:** Create or update `tests/test_fibo.py` in our practice folder.

**Why:** Include empty and short sequences and an invalid input instead of checking only a typical case.

**Expected result:** Four input rows and one exception test should pass.

Run this cell to save the file. The following test cell executes it. `%%writefile` is a notebook command: the lines below it become the contents of the named file.

In [ ]:
%%writefile tests/test_fibo.py
# Import pytest so we can use fixtures, marks, and exception checks.
import pytest
# Import the function or class that this test needs.
from fibo import fibo

# Run the same test separately for every input row below.
@pytest.mark.parametrize("n,expected", [(0, []), (1, [0]), (2, [0, 1]), (6, [0, 1, 1, 2, 3, 5])])
# Pytest discovers this test because its name starts with test_.
def test_fibonacci(n, expected):
    # Compare actual behavior with the expected result; a mismatch fails the test.
    assert fibo(n) == expected

# Pytest discovers this test because its name starts with test_.
def test_negative_count():
    # Pass only if the code inside raises the expected exception.
    with pytest.raises(ValueError, match="non-negative"):
        fibo(-1)


**What:** Run the saved test files.

**Why:** Writing a test file alone does not execute its tests. This command asks pytest to collect and run them.

**Expected result:** A successful run. With `--collect-only`, tests are listed without being executed. With a selection option, only matching tests run.

In [ ]:
# Execute pytest in a fresh process and check its exit code.
run_tests('tests/test_fibo.py', expected=0, options=())

## 4. Assertion differences (slides 15–19)
These tests deliberately compare different strings, lists, and dictionaries. Inspect the character differences, first unequal list index, and dictionary value. `-v` / `-vv` can reveal more detail.

### Compare unequal text, lists, and dictionaries

**What:** Create or update `demos/test_differences.py` in our practice folder.

**Why:** Pytest supplies useful differences without requiring custom diagnostic printing.

**Expected result:** Four intentional failures. Read the differing words, list index, and dictionary value.

Run this cell to save the file. The following test cell executes it. `%%writefile` is a notebook command: the lines below it become the contents of the named file.

In [ ]:
%%writefile demos/test_differences.py
# Pytest discovers this test because its name starts with test_.
def test_short_text():
    # Compare actual behavior with the expected result; a mismatch fails the test.
    assert "warrior" == "sorcerer"

# Pytest discovers this test because its name starts with test_.
def test_long_text():
    actual = "A battle-hardened veteran, favors heavy armor and weapons."
    expected = "A battle-hardened veteran, can equip heavy armor and weapons."
    # Compare actual behavior with the expected result; a mismatch fails the test.
    assert actual == expected

# Pytest discovers this test because its name starts with test_.
def test_list_difference():
    # Compare actual behavior with the expected result; a mismatch fails the test.
    assert [75, 80, 90] == [75, 85, 90]

# Pytest discovers this test because its name starts with test_.
def test_dictionary_difference():
    # Compare actual behavior with the expected result; a mismatch fails the test.
    assert {"name": "Solaire", "class": "warrior"} == {"name": "Solaire", "class": "sorcerer"}


**What:** Run the saved test files.

**Why:** Writing a test file alone does not execute its tests. This command asks pytest to collect and run them.

**Expected result:** A deliberate failure or setup error. Read the report. The notebook continues because this outcome is expected.

In [ ]:
# Execute pytest in a fresh process and check its exit code.
run_tests('demos/test_differences.py', expected=1, options=('-vv',))

**Practice:** Identify exactly which item differs in each case. A useful fix starts from the agreed requirement; do not automatically change an assertion merely to make it pass.

## 5. Checking exceptions and Activity 2 (slides 20–22)
The PPT’s character example rejects empty names and unsupported classes. Supporting exception classes and a return value are added to complete the executable example. `pytest.raises` checks the exception type; `match` checks its message using a regular expression.

### Create the character-validation functions

**What:** Create or update `characters.py` in our practice folder.

**Why:** Separate exception types make it clear which input rule was violated.

**Expected result:** An empty name or unsupported class raises an exception; a valid character returns a dictionary.

Run this cell to save the file. The following test cell executes it. `%%writefile` is a notebook command: the lines below it become the contents of the named file.

In [ ]:
%%writefile characters.py
class InvalidCharacterNameError(ValueError):
    pass

class InvalidClassNameError(ValueError):
    pass

VALID_CLASSES = ["sorcerer", "warrior"]

def create_character(name, class_name):
    if not name:
        # Reject invalid input with a clear exception.
        raise InvalidCharacterNameError("character name empty")
    if class_name not in VALID_CLASSES:
        # Reject invalid input with a clear exception.
        raise InvalidClassNameError(f'invalid class name: "{class_name}"')
    return {"name": name, "class": class_name}


### Inspect the wrong exception expectation in Activity 2

**What:** Create or update `demos/test_exceptions_wrong.py` in our practice folder.

**Why:** A test must expect the exception specified for that particular invalid input.

**Expected result:** The test fails: an unsupported class raises InvalidClassNameError, not InvalidCharacterNameError.

Run this cell to save the file. The following test cell executes it. `%%writefile` is a notebook command: the lines below it become the contents of the named file.

In [ ]:
%%writefile demos/test_exceptions_wrong.py
# Import pytest so we can use fixtures, marks, and exception checks.
import pytest
# Import the function or class that this test needs.
from characters import create_character, InvalidCharacterNameError

# Pytest discovers this test because its name starts with test_.
def test_invalid_class_name():
    # Pass only if the code inside raises the expected exception.
    with pytest.raises(InvalidCharacterNameError):
        create_character(name="Solaire", class_name="mage")


**What:** Run the saved test files.

**Why:** Writing a test file alone does not execute its tests. This command asks pytest to collect and run them.

**Expected result:** A deliberate failure or setup error. Read the report. The notebook continues because this outcome is expected.

In [ ]:
# Execute pytest in a fresh process and check its exit code.
run_tests('demos/test_exceptions_wrong.py', expected=1, options=())

**Activity 2:** Correct the exception type above. Add a test for an empty name and check the messages. Why should invalid name and invalid class have separate tests? They validate different rules.

### Correct Activity 2 and check valid behavior

**What:** Create or update `tests/test_characters.py` in our practice folder.

**Why:** Validate both error types and messages, then check the normal result.

**Expected result:** All three tests should pass.

Run this cell to save the file. The following test cell executes it. `%%writefile` is a notebook command: the lines below it become the contents of the named file.

In [ ]:
%%writefile tests/test_characters.py
# Import pytest so we can use fixtures, marks, and exception checks.
import pytest
# Import the function or class that this test needs.
from characters import create_character, InvalidCharacterNameError, InvalidClassNameError

# Pytest discovers this test because its name starts with test_.
def test_empty_name():
    # Pass only if the code inside raises the expected exception.
    with pytest.raises(InvalidCharacterNameError, match="character name empty"):
        create_character(name="", class_name="warrior")

# Pytest discovers this test because its name starts with test_.
def test_invalid_class_name():
    # Pass only if the code inside raises the expected exception.
    with pytest.raises(InvalidClassNameError, match='invalid class name: "mage"'):
        create_character(name="Solaire", class_name="mage")

# Pytest discovers this test because its name starts with test_.
def test_valid_character():
    # Compare actual behavior with the expected result; a mismatch fails the test.
    assert create_character("Solaire", "warrior") == {"name": "Solaire", "class": "warrior"}


**What:** Run the saved test files.

**Why:** Writing a test file alone does not execute its tests. This command asks pytest to collect and run them.

**Expected result:** A successful run. With `--collect-only`, tests are listed without being executed. With a selection option, only matching tests run.

In [ ]:
# Execute pytest in a fresh process and check its exit code.
run_tests('tests/test_characters.py', expected=0, options=())

## 6. Marks, parameters and configuration (slides 23–27)
A custom mark adds metadata. Register custom marks in `pytest.ini`; strict markers catch misspellings. `-m` selects by marker expression. `-k` selects by test name.

`@pytest.mark.timeout(10, method="thread")` comes from **pytest-timeout**. Registering a mark alone does not implement timeout behavior. The setup installs the plugin, and the helper explicitly loads it. The passing example below completes well before its limit; the thread method may terminate the subprocess if the limit is exceeded.

### Mark individual tests and a test class

**What:** Create or update `tests/test_marks.py` in our practice folder.

**Why:** Markers help run selected categories; the timeout plugin supplies actual time enforcement.

**Expected result:** Four tests should pass. The serial mark labels tests; it does not itself implement a scheduling policy.

Run this cell to save the file. The following test cell executes it. `%%writefile` is a notebook command: the lines below it become the contents of the named file.

In [ ]:
%%writefile tests/test_marks.py
# Import pytest so we can use fixtures, marks, and exception checks.
import pytest

# Attach a marker so this test or class can be selected with -m.
@pytest.mark.smoke
# Pytest discovers this test because its name starts with test_.
def test_quick_addition():
    # Compare actual behavior with the expected result; a mismatch fails the test.
    assert 2 + 2 == 4

# Attach a marker so this test or class can be selected with -m.
@pytest.mark.slow
# Limit execution time using the pytest-timeout plugin.
@pytest.mark.timeout(10, method="thread")
# Pytest discovers this test because its name starts with test_.
def test_long_computation():
    # Compare actual behavior with the expected result; a mismatch fails the test.
    assert sum(range(10000)) == 49995000

# Attach a marker so this test or class can be selected with -m.
@pytest.mark.serial
# Group related tests; pytest discovers classes starting with Test.
class TestCore:
    # Pytest discovers this test because its name starts with test_.
    def test_simple_simulation(self):
        # Compare actual behavior with the expected result; a mismatch fails the test.
        assert max([1, 4, 2]) == 4

    # Pytest discovers this test because its name starts with test_.
    def test_compute_tracers(self):
        # Compare actual behavior with the expected result; a mismatch fails the test.
        assert len(["a", "b"]) == 2


### Apply one marker to a complete module

**What:** Create or update `tests/test_module_marks.py` in our practice folder.

**Why:** Module metadata avoids repeating the same decorator on every test.

**Expected result:** The module test is selected by -m smoke.

Run this cell to save the file. The following test cell executes it. `%%writefile` is a notebook command: the lines below it become the contents of the named file.

In [ ]:
%%writefile tests/test_module_marks.py
# Import pytest so we can use fixtures, marks, and exception checks.
import pytest
pytestmark = pytest.mark.smoke

# Pytest discovers this test because its name starts with test_.
def test_module_mark():
    # Compare actual behavior with the expected result; a mismatch fails the test.
    assert "pytest".upper() == "PYTEST"


**What:** Run the saved test files.

**Why:** Writing a test file alone does not execute its tests. This command asks pytest to collect and run them.

**Expected result:** A successful run. With `--collect-only`, tests are listed without being executed. With a selection option, only matching tests run.

In [ ]:
# Execute pytest in a fresh process and check its exit code.
run_tests('tests', expected=0, options=('-q', '-m', 'smoke'))

**What:** Run the saved test files.

**Why:** Writing a test file alone does not execute its tests. This command asks pytest to collect and run them.

**Expected result:** A successful run. With `--collect-only`, tests are listed without being executed. With a selection option, only matching tests run.

In [ ]:
# Execute pytest in a fresh process and check its exit code.
run_tests('tests', expected=0, options=('-q', '-m', 'not slow and not serial'))

**What:** Run the saved test files.

**Why:** Writing a test file alone does not execute its tests. This command asks pytest to collect and run them.

**Expected result:** A successful run. With `--collect-only`, tests are listed without being executed. With a selection option, only matching tests run.

In [ ]:
# Execute pytest in a fresh process and check its exit code.
run_tests('tests/test_marks.py', expected=0, options=('-v',))

### Parametrization extension
The slides show arguments supplied to marks. `parametrize` goes further: it creates a separate test case for each input row. This is an added runnable example.

### Try several input rows with parametrization

**What:** Create or update `tests/test_parameters.py` in our practice folder.

**Why:** One test definition can check multiple scenarios while reporting each separately.

**Expected result:** Three addition cases should pass.

Run this cell to save the file. The following test cell executes it. `%%writefile` is a notebook command: the lines below it become the contents of the named file.

In [ ]:
%%writefile tests/test_parameters.py
# Import pytest so we can use fixtures, marks, and exception checks.
import pytest

# Run the same test separately for every input row below.
@pytest.mark.parametrize("a,b,expected", [(2, 3, 5), (0, 0, 0), (-1, 1, 0)])
# Pytest discovers this test because its name starts with test_.
def test_addition(a, b, expected):
    # Compare actual behavior with the expected result; a mismatch fails the test.
    assert a + b == expected


**What:** Run the saved test files.

**Why:** Writing a test file alone does not execute its tests. This command asks pytest to collect and run them.

**Expected result:** A successful run. With `--collect-only`, tests are listed without being executed. With a selection option, only matching tests run.

In [ ]:
# Execute pytest in a fresh process and check its exit code.
run_tests('tests/test_parameters.py', expected=0, options=('-v',))

**Practice:** Add a smoke mark to the Fibonacci test and select it with `-m smoke`. How does `pytestmark` differ from a decorator on a single function?

## 7. Fixtures (slides 28–31)
Fixtures provide reusable setup. Tests request fixtures by parameter name. Function scope, the default, runs the fixture separately for each test. A newly created list is therefore independent for each test. A fixture returning the same global mutable object would still share that object.

The series values below reproduce the PPT’s fixture example as training data.

### Use a comedy-series fixture

**What:** Create or update `tests/test_comedy.py` in our practice folder.

**Why:** Each test gets a newly created list, so mutating it in one test does not affect another.

**Expected result:** Highest rated: Seinfeld (8.9). Oldest: Seinfeld (1989). All four tests pass.

Run this cell to save the file. The following test cell executes it. `%%writefile` is a notebook command: the lines below it become the contents of the named file.

In [ ]:
%%writefile tests/test_comedy.py
# Import pytest so we can use fixtures, marks, and exception checks.
import pytest

# Turn this function into reusable test setup; default scope is one test.
@pytest.fixture
def comedy_series():
    return [
        ("The Office", 2005, 8.8),
        ("Scrubs", 2001, 8.4),
        ("IT Crowd", 2006, 8.5),
        ("Parks and Recreation", 2009, 8.6),
        ("Seinfeld", 1989, 8.9),
    ]

# Pytest discovers this test because its name starts with test_.
def test_highest_rated(comedy_series):
    # Compare actual behavior with the expected result; a mismatch fails the test.
    assert max(comedy_series, key=lambda item: item[2])[0] == "Seinfeld"

# Pytest discovers this test because its name starts with test_.
def test_oldest(comedy_series):
    # Compare actual behavior with the expected result; a mismatch fails the test.
    assert min(comedy_series, key=lambda item: item[1])[0] == "Seinfeld"

# Pytest discovers this test because its name starts with test_.
def test_mutation(comedy_series):
    comedy_series.clear()
    # Compare actual behavior with the expected result; a mismatch fails the test.
    assert comedy_series == []

# Pytest discovers this test because its name starts with test_.
def test_fresh_fixture(comedy_series):
    # Compare actual behavior with the expected result; a mismatch fails the test.
    assert len(comedy_series) == 5


**What:** Run the saved test files.

**Why:** Writing a test file alone does not execute its tests. This command asks pytest to collect and run them.

**Expected result:** A successful run. With `--collect-only`, tests are listed without being executed. With a selection option, only matching tests run.

In [ ]:
# Execute pytest in a fresh process and check its exit code.
run_tests('tests/test_comedy.py', expected=0, options=())

### Activity 3: Test class for drama series (slide 30)
Use the exact supplied data. Write a fixture named `drama_series`, then a class `TestDramaSeries` with tests for highest rating and oldest release year. Expected results: **Game of Thrones** (9.5) and **Cosmos** (1980).

Put the fixture in `tests/conftest.py` to share it across modules. Do not import it into the test module.

In [ ]:
# Exercise workspace: uncomment, complete, and run before viewing the solution.
# write_file("practice/test_drama.py", """
# import pytest
# @pytest.fixture
# def drama_series():
#     return [("The Mentalist", 2008, 8.1), ("Game of Thrones", 2011, 9.5),
#             ("The Newsroom", 2012, 8.6), ("Cosmos", 1980, 9.3)]
# class TestDramaSeries:
#     def test_highest_rated(self, drama_series):
#         raise NotImplementedError("Complete the test")
#     def test_oldest(self, drama_series):
#         raise NotImplementedError("Complete the test")
# """)
# run_tests("practice/test_drama.py", expected=1)

### Activity 3 solution: shared fixture

### Share fixtures and demonstrate cleanup

**What:** Create or update `tests/conftest.py` in our practice folder.

**Why:** Pytest automatically loads conftest.py for tests in this directory and below.

**Expected result:** The drama fixture becomes available without an import. The file fixture closes its handle after use.

Run this cell to save the file. The following test cell executes it. `%%writefile` is a notebook command: the lines below it become the contents of the named file.

In [ ]:
%%writefile tests/conftest.py
# Import pytest so we can use fixtures, marks, and exception checks.
import pytest

# Turn this function into reusable test setup; default scope is one test.
@pytest.fixture
def drama_series():
    return [
        ("The Mentalist", 2008, 8.1),
        ("Game of Thrones", 2011, 9.5),
        ("The Newsroom", 2012, 8.6),
        ("Cosmos", 1980, 9.3),
    ]

# Turn this function into reusable test setup; default scope is one test.
@pytest.fixture
def notes_file(tmp_path):
    handle = (tmp_path / "notes.txt").open("w+", encoding="utf-8")
    # Give the resource to the test; execution resumes here for cleanup afterward.
    yield handle
    # Release the file resource after the test finishes.
    handle.close()


### Complete Activity 3 with a test class

**What:** Create or update `tests/test_drama.py` in our practice folder.

**Why:** Group checks for the same series dataset and request the shared fixture by parameter name.

**Expected result:** Highest rated: Game of Thrones. Oldest: Cosmos. Both tests pass.

Run this cell to save the file. The following test cell executes it. `%%writefile` is a notebook command: the lines below it become the contents of the named file.

In [ ]:
%%writefile tests/test_drama.py
# Group related tests; pytest discovers classes starting with Test.
class TestDramaSeries:
    # Pytest discovers this test because its name starts with test_.
    def test_highest_rated(self, drama_series):
        # Compare actual behavior with the expected result; a mismatch fails the test.
        assert max(drama_series, key=lambda item: item[2]) == ("Game of Thrones", 2011, 9.5)

    # Pytest discovers this test because its name starts with test_.
    def test_oldest(self, drama_series):
        # Compare actual behavior with the expected result; a mismatch fails the test.
        assert min(drama_series, key=lambda item: item[1]) == ("Cosmos", 1980, 9.3)


### Use the same fixture in a second module

**What:** Create or update `tests/test_shared_fixture.py` in our practice folder.

**Why:** This demonstrates the benefit of shared setup and safe resource handling.

**Expected result:** Two tests should pass; the notes file is closed during fixture cleanup.

Run this cell to save the file. The following test cell executes it. `%%writefile` is a notebook command: the lines below it become the contents of the named file.

In [ ]:
%%writefile tests/test_shared_fixture.py
# Pytest discovers this test because its name starts with test_.
def test_shared_data(drama_series):
    # Compare actual behavior with the expected result; a mismatch fails the test.
    assert len(drama_series) == 4

# Pytest discovers this test because its name starts with test_.
def test_resource_cleanup_example(notes_file):
    notes_file.write("training notes")
    notes_file.seek(0)
    # Compare actual behavior with the expected result; a mismatch fails the test.
    assert notes_file.read() == "training notes"


**What:** Run the saved test files.

**Why:** Writing a test file alone does not execute its tests. This command asks pytest to collect and run them.

**Expected result:** A successful run. With `--collect-only`, tests are listed without being executed. With a selection option, only matching tests run.

In [ ]:
# Execute pytest in a fresh process and check its exit code.
run_tests('tests/test_drama.py', expected=0, options=())

**What:** Run the saved test files.

**Why:** Writing a test file alone does not execute its tests. This command asks pytest to collect and run them.

**Expected result:** A successful run. With `--collect-only`, tests are listed without being executed. With a selection option, only matching tests run.

In [ ]:
# Execute pytest in a fresh process and check its exit code.
run_tests('tests/test_shared_fixture.py', expected=0, options=())

`yield` separates setup from cleanup; the file closes after the test. `tmp_path` supplies an isolated temporary directory. These cleanup examples extend the fixture slides.

## 8. Optional extension: mocking
This section supports your original requested topics. It is additional to the PPT. A mock supplies a controlled dependency result; patch where the function under test looks up that dependency.

### Create a function with an external dependency

**What:** Create or update `notifications.py` in our practice folder.

**Why:** We need to isolate notification logic without sending an actual email.

**Expected result:** Calling send_email directly raises an error until a real service is configured. Tests replace it with a mock.

Run this cell to save the file. The following test cell executes it. `%%writefile` is a notebook command: the lines below it become the contents of the named file.

In [ ]:
%%writefile notifications.py
def send_email(recipient, message):
    # Reject invalid input with a clear exception.
    raise RuntimeError("No email service configured")

def notify_completion(name, email):
    return send_email(email, f"{name}, your training is complete")


### Test notification logic using a mock

**What:** Create or update `tests/test_mocking.py` in our practice folder.

**Why:** Control the dependency result and verify the intended call without contacting a provider.

**Expected result:** One test should pass; no email is sent.

Run this cell to save the file. The following test cell executes it. `%%writefile` is a notebook command: the lines below it become the contents of the named file.

In [ ]:
%%writefile tests/test_mocking.py
# Import the function or class that this test needs.
from unittest.mock import patch
# Import the function or class that this test needs.
from notifications import notify_completion

# Pytest discovers this test because its name starts with test_.
def test_completion_notification():
    # Replace the dependency temporarily; no real service is called.
    with patch("notifications.send_email", return_value="queued") as sender:
        # Compare actual behavior with the expected result; a mismatch fails the test.
        assert notify_completion("Asha", "asha@example.com") == "queued"
        # Verify that the dependency received exactly the intended arguments once.
        sender.assert_called_once_with("asha@example.com", "Asha, your training is complete")


**What:** Run the saved test files.

**Why:** Writing a test file alone does not execute its tests. This command asks pytest to collect and run them.

**Expected result:** A successful run. With `--collect-only`, tests are listed without being executed. With a selection option, only matching tests run.

In [ ]:
# Execute pytest in a fresh process and check its exit code.
run_tests('tests/test_mocking.py', expected=0, options=())

No message is actually sent. This checks our function’s interaction with the service; a separate integration test would be needed to check a real provider.

## 9. Optional extension: TDD
Requirement: A score from 0–100 passes at 60 or above. Values outside that range raise `ValueError`. Assume numeric inputs. Write tests first, observe Red, implement Green, then refactor and rerun the same tests.

### Implement or revise the assessment function

**What:** Create or update `assessment.py` in our practice folder.

**Why:** A TDD cycle starts with unfinished behavior, then implements and refactors it.

**Expected result:** The first version fails deliberately. Later versions pass the same boundary tests.

Run this cell to save the file. The following test cell executes it. `%%writefile` is a notebook command: the lines below it become the contents of the named file.

In [ ]:
%%writefile assessment.py
def is_pass(score):
    # Intentionally unfinished: this produces the Red step before implementation.
    raise NotImplementedError("Implement after tests")


### Write assessment tests before implementation

**What:** Create or update `tests/test_assessment.py` in our practice folder.

**Why:** Test the threshold at 60 and reject scores outside 0–100.

**Expected result:** The unfinished implementation fails six tests. The implemented and refactored versions pass all six.

Run this cell to save the file. The following test cell executes it. `%%writefile` is a notebook command: the lines below it become the contents of the named file.

In [ ]:
%%writefile tests/test_assessment.py
# Import pytest so we can use fixtures, marks, and exception checks.
import pytest
# Import the function or class that this test needs.
from assessment import is_pass

# Run the same test separately for every input row below.
@pytest.mark.parametrize("score,expected", [(0, False), (59, False), (60, True), (100, True)])
# Pytest discovers this test because its name starts with test_.
def test_result(score, expected):
    # Compare actual behavior with the expected result; a mismatch fails the test.
    assert is_pass(score) is expected

# Run the same test separately for every input row below.
@pytest.mark.parametrize("score", [-1, 101])
# Pytest discovers this test because its name starts with test_.
def test_invalid_score(score):
    # Pass only if the code inside raises the expected exception.
    with pytest.raises(ValueError, match="between 0 and 100"):
        is_pass(score)


**What:** Run the saved test files.

**Why:** Writing a test file alone does not execute its tests. This command asks pytest to collect and run them.

**Expected result:** A deliberate failure or setup error. Read the report. The notebook continues because this outcome is expected.

In [ ]:
# Execute pytest in a fresh process and check its exit code.
run_tests('tests/test_assessment.py', expected=1, options=())

### Implement or revise the assessment function

**What:** Create or update `assessment.py` in our practice folder.

**Why:** A TDD cycle starts with unfinished behavior, then implements and refactors it.

**Expected result:** The first version fails deliberately. Later versions pass the same boundary tests.

Run this cell to save the file. The following test cell executes it. `%%writefile` is a notebook command: the lines below it become the contents of the named file.

In [ ]:
%%writefile assessment.py
def is_pass(score):
    if score < 0 or score > 100:
        # Reject invalid input with a clear exception.
        raise ValueError("score must be between 0 and 100")
    return score >= 60


**What:** Run the saved test files.

**Why:** Writing a test file alone does not execute its tests. This command asks pytest to collect and run them.

**Expected result:** A successful run. With `--collect-only`, tests are listed without being executed. With a selection option, only matching tests run.

In [ ]:
# Execute pytest in a fresh process and check its exit code.
run_tests('tests/test_assessment.py', expected=0, options=())

### Refactor
Name the pass threshold without changing behavior. Existing tests protect the boundary at exactly 60.

### Implement or revise the assessment function

**What:** Create or update `assessment.py` in our practice folder.

**Why:** A TDD cycle starts with unfinished behavior, then implements and refactors it.

**Expected result:** The first version fails deliberately. Later versions pass the same boundary tests.

Run this cell to save the file. The following test cell executes it. `%%writefile` is a notebook command: the lines below it become the contents of the named file.

In [ ]:
%%writefile assessment.py
PASS_THRESHOLD = 60

def is_pass(score):
    if not 0 <= score <= 100:
        # Reject invalid input with a clear exception.
        raise ValueError("score must be between 0 and 100")
    return score >= PASS_THRESHOLD


**What:** Run the saved test files.

**Why:** Writing a test file alone does not execute its tests. This command asks pytest to collect and run them.

**Expected result:** A successful run. With `--collect-only`, tests are listed without being executed. With a selection option, only matching tests run.

In [ ]:
# Execute pytest in a fresh process and check its exit code.
run_tests('tests/test_assessment.py', expected=0, options=())

## 10. Final suite and learner checklist
The completed suite excludes deliberate failures in `demos` and optional work in `practice`. Running without a path uses the configured `tests` folder. All completed tests should pass.

**What:** Run the saved test files.

**Why:** Writing a test file alone does not execute its tests. This command asks pytest to collect and run them.

**Expected result:** A successful run. With `--collect-only`, tests are listed without being executed. With a selection option, only matching tests run.

In [ ]:
# Execute pytest in a fresh process and check its exit code.
run_tests(None, expected=0, options=('-q',))

**What:** Run the saved test files.

**Why:** Writing a test file alone does not execute its tests. This command asks pytest to collect and run them.

**Expected result:** A successful run. With `--collect-only`, tests are listed without being executed. With a selection option, only matching tests run.

In [ ]:
# Execute pytest in a fresh process and check its exit code.
run_tests('tests', expected=0, options=('--collect-only', '-q'))

Before finishing, confirm that you can:
- Explain test discovery and interpret a traceback.
- Correct the Fibonacci and exception activities from their requirements.
- Select tests by marker and register custom marks.
- Use parametrization for input cases and boundaries.
- Request a shared fixture from a test class and explain its scope.
- Distinguish unit checks with mocks from real integration checks.
- Follow a small Red → Green → Refactor cycle.

**Best practices:** Keep tests independent, use meaningful names, test boundaries and errors, isolate external dependencies, and rerun relevant tests after refactoring. Passing tests prove only the tested cases.

**Troubleshooting:** Rerun file-writing cells after edits. After a Colab runtime restart, rerun setup. Exit 0 means success, 1 means tests failed, 2 often indicates interrupted collection, and 5 means no tests were collected. Expected-failure demonstrations use `expected=1`; use the default `expected=0` after fixing them.

**Sources:** Supplied `2. PyTest.pptx`; [pytest discovery](https://docs.pytest.org/en/stable/explanation/goodpractices.html), [markers](https://docs.pytest.org/en/stable/example/markers.html), [fixtures](https://docs.pytest.org/en/stable/how-to/fixtures.html), [parametrization](https://docs.pytest.org/en/stable/how-to/parametrize.html), [pytest-timeout](https://github.com/pytest-dev/pytest-timeout), [unittest.mock](https://docs.python.org/3/library/unittest.mock.html).
